In [24]:
import pandas as pd
import json

In [ ]:
def generar_dataset_agente():
    print("Cargando archivos CSV...")
    
    # 1. Cargar los 3 archivos CSV
    # Ajusta las rutas según dónde tengas guardados los archivos en tu proyecto
    df_metricas = pd.read_csv("estadisticas_notas.csv")
    df_catalogo = pd.read_csv("info_notas.csv", engine='python', on_bad_lines='skip')
    df_suscripciones = pd.read_csv("sus_notas.csv")
    
    # 2. Homogeneizar tipos de datos para asegurar el cruce correcto (convertir IDs a String y quitar espacios)
    df_metricas['nota_id'] = df_metricas['nota_id'].astype(str).str.strip()
    df_catalogo['nota_id'] = df_catalogo['nota_id'].astype(str).str.strip()
    
    # En el archivo de BQ la columna se llama 'id_nota'
    df_suscripciones['id_nota'] = df_suscripciones['id_nota'].astype(str).str.strip()
    
    print("Realizando el cruce de datos (Merging)...")
    
    # 3. Unir Catálogo con Métricas
    df_unificado = pd.merge(df_catalogo, df_metricas, on='nota_id', how='inner')
    
    # 4. Unir con Suscripciones (usando left join para mantener en 0 las que no tengan suscripciones)
    df_unificado = pd.merge(df_unificado, df_suscripciones, left_on='nota_id', right_on='id_nota', how='left')
    
    # 5. Rellenar con 0 aquellas notas que no tuvieron ninguna suscripción registrada
    df_unificado['total_suscripciones'] = df_unificado['total_suscripciones'].fillna(0).astype(int)
    
    print("Estructurando los datos al formato JSON del Agente...")
    
    registros_json = []
    
    # 6. Iterar sobre las filas para construir la estructura jerárquica exacta solicitada
    for _, fila in df_unificado.iterrows():
        estructura_nodo = {
            "nota_id": str(fila['nota_id']),
            "titulo": str(fila['titulo']),
            "seccion": str(fila['seccion']),
            "tipo_acceso": str(fila['acceso']),
            "metricas": {
                "vistas": int(fila['screenPageViews']),
                "ratio_fidelidad_dau_mau": float(fila['dauPerMau']),
                "calidad_iql_consumo": float(fila['iql1']),
                "calidad_iql_interaccion": float(fila['iql2']),
                "suscripciones_generadas": int(fila['total_suscripciones'])
            }
        }
        registros_json.append(estructura_nodo)
        
    # 7. Guardar el resultado en formato JSONL (JSON Lines), que es el formato estándar para Vertex AI Fine-Tuning
    ruta_salida = "dataset_agente_editorial.jsonl"
    with open(ruta_salida, 'w', encoding='utf-8') as f:
        for reg in registros_json:
            f.write(json.dumps(reg, ensure_ascii=False) + '\n')
            
    print(f"¡Proceso completado con éxito! Se han procesado {len(registros_json)} registros.")
    print(f"Archivo guardado en: {ruta_salida}")
    
    # Imprimir una muestra en la consola de VS Code para verificar el formato
    if registros_json:
        print("\nMuestra del primer registro generado:")
        print(json.dumps(registros_json[0], indent=2, ensure_ascii=False))

if __name__ == "__main__":
    generar_dataset_agente()

Cargando archivos CSV...
Realizando el cruce de datos (Merging)...
Estructurando los datos al formato JSON del Agente...
¡Proceso completado con éxito! Se han procesado 122591 registros.
Archivo guardado en: dataset_agente_editorial.jsonl

Muestra del primer registro generado:
{
  "nota_id": "1020704",
  "titulo": "Aguas infusionadas: la tendencia en bebidas frescas y saludables para el verano",
  "seccion": "Sociedad",
  "tipo_acceso": "Medido",
  "metricas": {
    "vistas": 12,
    "ratio_fidelidad_dau_mau": 1.0,
    "calidad_iql_consumo": 0.0489177,
    "calidad_iql_interaccion": 0.119203,
    "suscripciones_generadas": 0
  }
}


In [6]:
import pandas as pd
import json

def generar_dataset_historico():
    print("Cargando archivo CSV histórico completo (2024-2025)...")
    
    # 1. Cargar el dataset con protección para comas internas en títulos
    try:
        df_hist = pd.read_csv(
            "notas_metricas_historicas.csv", 
            engine='python', 
            on_bad_lines='skip'
        )
    except FileNotFoundError:
        print("Error: No se encontró el archivo 'notas_metricas_historicas.csv'")
        return

    # 2. Homogeneizar y limpiar variables cualitativas clave en la raíz
    df_hist['nota_id'] = df_hist['nota_id'].astype(str).str.strip()
    df_hist['titulo'] = df_hist['titulo'].astype(str).str.strip() if 'titulo' in df_hist.columns else "Sin título"
    df_hist['seccion'] = df_hist['seccion'].astype(str).str.strip().str.capitalize()
    df_hist['fecha'] = df_hist['fecha'].astype(str).str.strip()
    
    print("Estructurando el 100% de las variables métricas en el formato JSON...")
    
    registros_hist_json = []
    
    # 3. Iterar mapeando absolutamente todo el set de datos disponible
    for _, fila in df_hist.iterrows():
        estructura_nodo = {
            "nota_id": str(fila['nota_id']),
            "titulo": str(fila['titulo']),
            "seccion": str(fila['seccion']),
            "fecha_publicacion": str(fila['fecha']) if pd.notna(fila['fecha']) else "Sin Fecha",
            "tipo_acceso": str(fila['acceso']) if pd.notna(fila['acceso']) else "No medido",
            "origen_agenda": str(fila['origen']) if pd.notna(fila['origen']) else "Desconocido",
            "article_type": str(fila['article_type']) if pd.notna(fila['article_type']) else "news",
            "fuente_plataforma": str(fila['fuente']) if pd.notna(fila['fuente']) else "Online",
            "tipo_nota_cms": str(fila['tipo_nota']) if pd.notna(fila['tipo_nota']) else "Nota estándar",
            
            # Sub-nodo con todas las métricas de comportamiento e impacto político/UX
            "metricas_historicas": {
                # Volumen e Interacción General
                "paginas_vistas": int(fila['paginas_vistas']) if pd.notna(fila['paginas_vistas']) else 0,
                "visitas_totales": int(fila['visitas']) if pd.notna(fila['visitas']) else 0,
                "usuarios_unicos": int(fila['users']) if pd.notna(fila['users']) else 0,
                "sesiones_totales": int(fila['sessions']) if pd.notna(fila['sessions']) else 0,
                "compartidas_redes": int(fila['compartidas']) if pd.notna(fila['compartidas']) else 0,
                "comentarios_usuarios": int(fila['comentarios']) if pd.notna(fila['comentarios']) else 0,
                
                # Telemetría de Tiempos y Calidad (UX)
                "tiempo_total_paginas": float(fila['tiempo_paginas']) if pd.notna(fila['tiempo_paginas']) else 0.0,
                "tiempo_interaccion_engaged": float(fila['engaged_time']) if pd.notna(fila['engaged_time']) else 0.0,
                "tiempo_permanencia_spent": float(fila['time_spent']) if pd.notna(fila['time_spent']) else 0.0,
                "tasa_rebote_bounce": float(fila['rebote']) if pd.notna(fila['rebote']) else 0.0,
                "porcentaje_scroll_medio": float(fila['scroll']) if pd.notna(fila['scroll']) else 0.0,
                
                # Multimedia e Impacto de Video
                "tiene_fotos": int(fila['tiene_fotos']) if pd.notna(fila['tiene_fotos']) else 0,
                "tiene_video": int(fila['tiene_video']) if pd.notna(fila['tiene_video']) else 0,
                "visitas_reproduccion_video": int(fila['visitas_video']) if pd.notna(fila['visitas_video']) else 0,
                
                # Audiencia y Fidelidad
                "usuarios_fieles_loyal": int(fila['loyal']) if pd.notna(fila['loyal']) else 0,
                "lectores_people": int(fila['people']) if pd.notna(fila['people']) else 0,
                
                # Métricas Editoriales Avanzadas y Conversión
                "longitud_caracteres": int(fila['longitud']) if pd.notna(fila['longitud']) else 0,
                "score_sentimiento": str(fila['sentimiento']) if pd.notna(fila['sentimiento']) else 'None',
                "conversiones_suscripcion": int(fila['conversion']) if pd.notna(fila['conversion']) else 0,
                
                # Indicadores de Lectura Final (Telemetría de Scroll de salida)
                "llego_nota_final": int(fila['leenotafinal']) if pd.notna(fila['leenotafinal']) else 0,
                "permanencia_final30_seg": int(fila['final30']) if pd.notna(fila['final30']) else 0
            }
        }
        registros_hist_json.append(estructura_nodo)
        
    # 4. Guardar el resultado en el archivo JSON Lines
    ruta_salida = "dataset_historico_agente.jsonl"
    with open(ruta_salida, 'w', encoding='utf-8') as f:
        for reg in registros_hist_json:
            f.write(json.dumps(reg, ensure_ascii=False) + '\n')
            
    print(f"¡Proceso completado! Se exportaron todas las variables de {len(registros_hist_json)} registros antiguos.")
    print(f"Archivo histórico guardado en: {ruta_salida}")

if __name__ == "__main__":
    generar_dataset_historico()

Cargando archivo CSV histórico completo (2024-2025)...
Estructurando el 100% de las variables métricas en el formato JSON...
¡Proceso completado! Se exportaron todas las variables de 56616 registros antiguos.
Archivo histórico guardado en: dataset_historico_agente.jsonl


In [7]:
import pandas as pd
import json

print("Iniciando consolidación de la Matriz de Conocimiento Avanzada...")

# ==========================================
# 1. PROCESAR DATASET ACTUAL (PRODUCCIÓN)
# ==========================================
df_actual = pd.read_json("dataset_agente_editorial.jsonl", lines=True)

# Desanidar las métricas actuales nativas
df_actual['vistas'] = df_actual['metricas'].apply(lambda x: x.get('vistas', 0))
df_actual['iql_interaccion'] = df_actual['metricas'].apply(lambda x: x.get('calidad_iql_interaccion', 0))
df_actual['suscripciones'] = df_actual['metricas'].apply(lambda x: x.get('suscripciones_generadas', 0))

# Agrupación y promedios del estado actual
resumen_actual = df_actual.groupby('seccion').agg(
    notas_publicadas_actual=('nota_id', 'count'),
    vistas_promedio_actual=('vistas', 'mean'),
    iql_interaccion_promedio_actual=('iql_interaccion', 'mean'),
    total_suscripciones_actual=('suscripciones', 'sum')
).reset_index()


# ==========================================
# 2. PROCESAR DATASET HISTÓRICO COMPLETO (2024-2025)
# ==========================================
df_hist = pd.read_json("dataset_historico_agente.jsonl", lines=True)

# Desanidar el 100% de las nuevas métricas históricas integradas
df_hist['vistas_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('paginas_vistas', 0))
df_hist['scroll_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('porcentaje_scroll_medio', 0.0))
df_hist['engaged_time_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('tiempo_interaccion_engaged', 0.0))
df_hist['loyal_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('usuarios_fieles_loyal', 0))
df_hist['tiene_video_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('tiene_video', 0))
df_hist['compartidas_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('compartidas_redes', 0))
df_hist['suscripciones_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('conversiones_suscripcion', 0))
df_hist['longitud_hist'] = df_hist['metricas_historicas'].apply(lambda x: x.get('longitud_caracteres', 0))

# Agrupación avanzada del comportamiento histórico por sección
resumen_historico = df_hist.groupby('seccion').agg(
    notas_publicadas_hist=('nota_id', 'count'),
    vistas_promedio_hist=('vistas_hist', 'mean'),
    scroll_promedio_hist=('scroll_hist', 'mean'),
    tiempo_interaccion_promedio_hist=('engaged_time_hist', 'mean'),
    usuarios_fieles_promedio_hist=('loyal_hist', 'mean'),
    porcentaje_notas_con_video_hist=('tiene_video_hist', 'mean'), # Qué porcentaje de notas llevaba video
    compartidas_promedio_hist=('compartidas_hist', 'mean'),
    longitud_caracteres_promedio_hist=('longitud_hist', 'mean'),
    total_suscripciones_hist=('suscripciones_hist', 'sum')
).reset_index()


# ==========================================
# 3. UNIFICACIÓN Y DISCRIMINACIÓN DE ERAS
# ==========================================
# Combinamos ambos dataframes usando un Outer Join para no perder ninguna sección
matriz_combinada = pd.merge(resumen_actual, resumen_historico, on='seccion', how='outer').fillna(0)

# Convertir a un formato de diccionario indexado por sección para consumo del Agente
matriz_conocimiento = matriz_combinada.set_index('seccion').to_dict(orient='index')

# Opcional: Guardarla a un archivo físico para que tu app.py la lea de forma veloz
with open("matriz_conocimiento_editorial.json", "w", encoding="utf-8") as f:
    json.dump(matriz_conocimiento, f, ensure_ascii=False, indent=4)

print("¡Matriz de conocimiento combinada creada con éxito discriminando ambos datasets!")

Iniciando consolidación de la Matriz de Conocimiento Avanzada...
¡Matriz de conocimiento combinada creada con éxito discriminando ambos datasets!


In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
model = genai.GenerativeModel('gemini-2.5-flash') # Usamos Pro para un análisis correlacional más profundo

def evaluar_potencial_nota(nuevo_titulo, nueva_seccion):
    
    # Este string contiene el resumen estadístico real de nuestros 122k notas
    # (Para el ejemplo lo pongo hardcodeado, pero idealmente pasas la variable del Paso 1)
    contexto_datos = """
    A continuación tienes el rendimiento histórico resumido de nuestra plataforma:
    - Sección 'Sociedad': Alta densidad de notas, vistas promedio moderadas (12-15), iql_interaccion alto en temas de tendencias de verano, suscripciones bajas (0-1 por nota).
    - Sección 'Opinión': Volumen bajo de notas, vistas bajas, pero iql_interaccion crítico (0.95) y alta tasa de conversión a suscripciones en temas políticos (Manzur, Jaldo).
    - Sección 'Economía': Vistas volátiles, engagement rate alto si el título incluye palabras de impacto como 'Dólar', 'Inflación', 'Impuestos'.
    """
    
    prompt = f"""
    Eres un Agente de Inteligencia Artificial experto en Analítica de Medios y Estrategia Editorial.
    Tu objetivo es evaluar el potencial de éxito de una propuesta de nota antes de ser escrita, basándote en la correlación histórica de nuestros datos.
    
    {contexto_datos}
    
    ---
    PROPUESTA A EVALUAR:
    Título de la nota: "{nuevo_titulo}"
    Categoría/Sección: "{nueva_seccion}"
    ---
    
    Por favor, genera un informe analítico detallado que incluya:
    1. **Correlación Histórica**: ¿Cómo se alinea este título con el rendimiento típico de la sección? (Analiza palabras clave del título vs la categoría).
    2. **Predicción de Rendimiento**: ¿Qué se espera a nivel de Vistas (Alto/Medio/Bajo), Calidad de Lectura (IQL) y Propensión a generar Suscripciones?
    3. **Recomendaciones de Optimización**: Qué palabras clave agregar al título o qué enfoque darle al contenido dentro de esa categoría para maximizar su IQL y conversiones.
    """
    
    response = model.generate_content(prompt)
    return response.text

# --- PRUEBA DEL ESCENARIO ---
if __name__ == "__main__":
    titulo_propuesto = "El impacto de las nuevas tarifas de luz en los comercios locales"
    seccion_propuesta = "Economía"
    
    print("Evaluando propuesta con el agente...")
    analisis = evaluar_potencial_nota(titulo_propuesto, seccion_propuesta)
    print(" " + analisis)


Evaluando propuesta con el agente...

¡Excelente propuesta para analizar! Como Agente de Inteligencia Artificial experto en Analítica de Medios y Estrategia Editorial, procedo a evaluar el potencial de éxito de la nota "El impacto de las nuevas tarifas de luz en los comercios locales" para la sección de "Economía", basándome en nuestra correlación histórica.

---

### Informe Analítico Detallado: Evaluación de Propuesta Editorial

**Propuesta a Evaluar:**
*   **Título:** "El impacto de las nuevas tarifas de luz en los comercios locales"
*   **Categoría/Sección:** "Economía"

---

#### 1. Correlación Histórica: Alineación con el Rendimiento Típico de la Sección

La propuesta se ubica en la sección de "Economía", la cual, según nuestros datos, presenta **vistas volátiles** pero un **engagement rate alto** si el título incluye palabras de impacto como 'Dólar', 'Inflación' o 'Impuestos'.

*   **Análisis de Palabras Clave del Título:**
    *   **"Impacto":** Esta palabra clave es poderosa e